# Homework: Build a StationXML Inventory for recent KSC deployment

Seven USF stations were deployed at Kennedy Space Center between January 7-12, 2026.
* Each features a Nanometrics Centaur digitizer, and a Nanometrics Trillium Compact Post-Hole 120-s seismometer.
* This is the same equipment we setup in our class a few weeks ago - except we also had infrasound.

Three additional seismic stations were setup by Marshall Space Flight Center, using Guralp and Silicon Audio equipment. You can ignore those for this exercise - or potentially earn some bonus points if you find responses for those too!

## Goal
Create a valid **StationXML** file for this small network using:
- Station coordinates stored in an **Excel** file
- Instrument responses pulled from the **IRIS Nominal Response Library (NRL)**

You will:
1. Load the station list with **pandas** into a pandas DataFrame
2. Create a Response object for a USF Nanometrics station
3. Build an ObsPy **Inventory**
4. Write out a **StationXML** file

> We're keeping it simple: one network, three channels per station (Z, N, E), same equipment at each site.

**Warning:** only attempt this homework after you have completed this week's reading exercises (notebooks 73 and 74)

## Provided file
Your station list is in:

- `Summary_Seismic_Station_List.xlsx`

One column contains photos — **ignore it**.

An alternative version of the Excel file is also provided as a Comma-Separated-Variable (CSV) text file, just in case you cannot read the Excel file. This is easier to read.

(If you open any Excel file with Excel, you can save it as a CSV).

## Make sure you have openpyxl installed in your conda environment!
This is required to read Excel files. CSV files will work without it.

In [ ]:
import sys
#!conda install -y openpyxl --prefix {sys.prefix}

## Minimal pandas intro (what you need today)

**pandas** is a Python library for working with tables (“dataframes”).  
For this homework you only need to:
- read an Excel file
- select a few columns
- loop over rows

**Note**: you may need to install openpyxl in conda, and then restart this notebook/kernel. At the command line:

```
conda activate compsci # or whatever your conda env is
conda install openpyxml
```

In [ ]:
import pandas as pd

xlsx_path = r"Summary_Seismic_Station_List.xlsx"
df = pd.read_excel(xlsx_path)

df.head(10)

If for any reason you cannot load the Excel file, you can try the CSV file instead:
```
df = pd.read_csv("Summary_Seismic_Station_List.csv")
```

In [ ]:
df.columns

## Step 1 — Keep only the columns we need

For StationXML, we need at minimum:
- station code (name)
- latitude
- longitude

We will also keep:
- seismometer model (if present)
- digitizer model (if present)

In [ ]:
use_cols = ["Site Name", "lat", "lon", "Seismometer", "Digitizer"]
stations = df[use_cols].copy()

# Ignore the Photo column (already excluded)
stations

## Step 2 — Create a Response object for a USF Nanometrics station

In [ ]:
# TO DO: code for Response object for USF Nanometrics station from NRL
from obspy.clients.nrl import NRL
from obspy.core.inventory import Inventory, Network, Station, Channel, Site
from obspy import UTCDateTime

# initialize
all_NRL = NRL()
print(all_NRL.sensors["Nanometrics"])
print(all_NRL.sensors["Nanometrics"]["Trillium Compact 120 (Vault, Posthole, OBS)"])
print('\nnext sensor\n')


In [ ]:
print(all_NRL.sensors.keys())
print(all_NRL.sensors["Guralp"].keys())
print(all_NRL.sensors["Silicon Audio"].keys())
#print(all_NRL.sensors["Chaparral Physics"].keys())
#print(all_NRL.sensors["Nanometrics"].keys())

# Note: appears that censors are NOT in the network...

In [ ]:

# look up USF sensors
# Nanometrics Centaur digitizer, and a Nanometrics Trillium Compact Post-Hole 120-s seismometer
USF_digitizer_keys = ["Nanometrics", "Centaur", "40 Vpp (1)", "Off", "Linear phase", "100"]
USF_sensor_keys = ["Nanometrics",'Trillium Compact 120 (Vault, Posthole, OBS)','754 V/m/s']
usf_combined_response = all_NRL.get_response(datalogger_keys = USF_digitizer_keys,sensor_keys=USF_sensor_keys)
usf_sensor_response, _ = all_NRL._get_response("sensors", keys=USF_sensor_keys)
usf_digitizer_response, _ = all_NRL._get_response("dataloggers", keys=USF_digitizer_keys)
print("Combined response:", usf_combined_response)

In [ ]:
# Kennedy Guralp and Silicon Audio equipment
# https://www.fdsn.org/station_book/ii.html

## Step 3 - Create an ObsPy Inventory

The Inventory object should contain a list of 1 Network object.

The Network object should have a network code, description, and start_date, and contain a list of 7 Station objects.

Each Station object should have a station code, description, and coordinates, and a list of 3 Channel objects.

Each Channel object should have a channel code, a location code, coordinates (StationXML repeats them), depth, and sampling rate, and a Response object.


In [ ]:
# TO DO: Here is some starter code for the Inventory object. You will need to update it with station metadata from the Excel file and Response metadata from NRL.

from obspy.core.inventory import Inventory, Network, Station, Channel, Site
from obspy import UTCDateTime
import numpy as np

# -----------------------------
# Network-level metadata
# -----------------------------
NETWORK_CODE = "1R"                     
NETWORK_DESCRIPTION = "KSC Seismic Network"
START_DATE = UTCDateTime(2026, 1, 7)    # Installation of USF sensors date

# -----------------------------
# Channel metadata
# -----------------------------
CHANNEL_CODES = ["DHZ", "DHN", "DHE"]   # Google "SEED channel naming"
LOC_CODE = ""                           # usually blank unless multiple sensors
FS = 500.0                              # sample rate (Hz)
ELEV_M = 0.0                            # good enough - KSC is close to sea level
DEPTH_M = 0.75                          # burial depth (m) - holes were 2-3 feet deep
SENSOR = "Nanometrics Trillium C-PH 120"             
DIGITIZER = "Nanometrics Centaur"  

# Create Response object for USF Nanometrics station from NRL
# created above
# print("Combined response:", usf_combined_response)

# Loop over rows in DataFrame - one row per station ("site name" is the station code)
stations = []
for index, row in df.iterrows():
    sta_code = row["Site Name"]
    latitude = row["lat"]
    longitude = row["lon"]
    print(row)
    if not pd.isna(row["Seismometer"]):
        myresp = usf_combined_response
    else:
        myresp = None

    # Loop over CHANNEL_CODES and create a Channel object for each one.
    channels = []
    for chan_code in CHANNEL_CODES:
        # fill in - create a new Channel object - make sure to include the Response object you created above
        # append to channels list
        ch = Channel(
            code = chan_code,
            location_code=LOC_CODE,
            latitude=latitude, 
            longitude=longitude,
            elevation=ELEV_M, 
            depth=DEPTH_M,
            sample_rate=FS,
            response=myresp
        )
        channels.append(ch)
    # Create station information
    station = Station(
        code = sta_code,
        latitude=latitude,
        longitude=longitude,
        elevation=ELEV_M,
        creation_date=START_DATE,
        site=Site(name=row.get("Site Name",sta_code)),
        channels=channels
    )
    # Add station to stations list
    stations.append(station)

# Create a Network object and add the stations to it. Then create an Inventory object and add the network to it.
# Create Network object
network = Network(
    code=NETWORK_CODE,
    description=NETWORK_DESCRIPTION,
    stations=stations,
    start_date=START_DATE
)

# Create Inventory object
inventory = Inventory(
    networks=[network],
    source="KSC Seismic Network Deployment"
)

# Check summary
print(inventory)

net = inventory.networks[0]
sta = net.stations[0]
chan = sta.channels[0]

# Verify response loaded in
#print(chan)  # prints channel metadata
#print(chan.response)  # prints Response object

## Step 4 — Build Inventory, plot, and write StationXML

In [ ]:
# TO DO

# PERFORMED ABOVE
# from obspy.core.inventory import Inventory
# fill in - create Inventory object and add Network object to it
inventory.plot();
inventory.plot_response(min_freq=0.001, output="VEL",station="B03",channel="DHZ");
inventory.plot_response(min_freq=0.001, output="DISP",station="B03",channel="DHZ");
inventory.write("ksc_inventory.xml", format="STATIONXML")

## Step 5 — Quick check: read back the StationXML

In [ ]:
from obspy import read_inventory

inv2 = read_inventory("ksc_inventory.xml")
print(inv2)

In [ ]:
stations_with_response = []
stations_missing_response = []

for sta in inventory.networks[0].stations:
    has_response = any(chan.response for chan in sta.channels)
    if has_response:
        stations_with_response.append(sta.code)
    else:
        stations_missing_response.append(sta.code)

print("=== NRL Response Summary ===")
print(f"Stations with NRL response ({len(stations_with_response)}): {stations_with_response}")
print(f"Stations missing NRL response ({len(stations_missing_response)}): {stations_missing_response}")


## What to submit

Submit your notebook **and** the generated StationXML file.

### Your short note (2–4 sentences)
At the end of your notebook, include a short note:
- Which stations successfully got responses from the NRL?
- Which stations did not, and why (missing mapping, missing instrument info, etc.)?

# add notes below...

All stations reported a response from NRL (except the Kennedy Space Center Stations).  These stations would need a response for their own instruments.  As I was not able to find the instrument info, I could not fill in the information.

ChatGPT was used to help develop this code and several checks along the way.  Most notably was working with a way to separate out the KSC instruments from the USF ones (which was done fairly lazily using a NaN value check).  It also wrote the code for step 5 as it could write the checks faster than myself.  Other code was sourced from within this workbook or from similar instructions from files 73 or 74.